# 🧠 DatasheetIQ — Knowledge Graph Backend (Google Colab)

Runs **FastAPI backend + Qwen 3.5 4B** in Google Colab with nohup.

**Run cells in order. Do NOT skip any cell.**

## ⚠️ PHASE 1 — Clean Environment

In [ ]:
# CELL 1: Clean conflicting packages
!pip uninstall -y fastapi starlette httpx uvicorn pydantic python-multipart 2>/dev/null
print("✅ Cleaned pre-installed packages")

## 📦 PHASE 2 — Install Dependencies

In [ ]:
# CELL 2: Core backend (PINNED)
!pip install fastapi==0.124.1 uvicorn==0.34.0 starlette==0.49.1 pydantic==2.12.0 pydantic-settings httpx==0.28.1 python-multipart==0.0.18
print("\n✅ Core backend installed")

In [ ]:
# CELL 3: PDF parsing + Neo4j + ngrok
!pip install PyMuPDF==1.24.0 pdfplumber==0.11.0 Pillow==10.4.0 neo4j==5.23.0 pyngrok
print("\n✅ PDF + Neo4j + ngrok installed")

In [ ]:
# CELL 4: AI/ML
!pip install transformers accelerate
print("\n✅ Transformers + Accelerate installed")

## ✅ PHASE 3 — Validate

In [ ]:
# CELL 5: Validate imports
import fastapi, uvicorn, pydantic, starlette, httpx
import fitz, pdfplumber, neo4j, torch, transformers

print(f"fastapi     : {fastapi.__version__}")
print(f"uvicorn     : {uvicorn.__version__}")
print(f"pydantic    : {pydantic.__version__}")
print(f"starlette   : {starlette.__version__}")
print(f"httpx       : {httpx.__version__}")
print(f"torch       : {torch.__version__}")
print(f"GPU         : {torch.cuda.is_available()} — {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print("\n✅ ALL IMPORTS VALID")

## 📂 PHASE 4 — Mount Drive + Set Path

In [ ]:
# CELL 6: Mount Google Drive and set project root
import os, sys

from google.colab import drive
drive.mount('/content/drive')

# ⚠️ This must point to the folder containing backend/
PROJECT_ROOT = '/content/drive/MyDrive/Knowledge_graph'

os.chdir(PROJECT_ROOT)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"✅ Working directory: {os.getcwd()}")
print(f"✅ sys.path[0]: {sys.path[0]}")

In [ ]:
# CELL 7: Verify project structure
import os

required = [
    'backend/__init__.py',
    'backend/main.py',
    'backend/app/__init__.py',
    'backend/app/main.py',
    'backend/app/config.py',
    'backend/app/models.py',
    'backend/app/routers/upload.py',
    'backend/app/routers/query.py',
    'backend/app/services/pdf_parser.py',
    'backend/app/services/table_extractor.py',
    'backend/app/services/graph_builder.py',
    'backend/app/services/query_engine.py',
    'backend/app/services/ai_client.py',
    'backend/app/utils/__init__.py',
]

missing = [f for f in required if not os.path.exists(f)]
if missing:
    print("❌ MISSING FILES:")
    for f in missing:
        print(f"   {f}")
else:
    print("✅ All files found")

# Verify FastAPI app is importable
try:
    from backend.main import app
    print(f"✅ 'from backend.main import app' works — type: {type(app).__name__}")
except Exception as e:
    print(f"❌ Import failed: {e}")

## 🔗 PHASE 5 — Neo4j Credentials

In [ ]:
# CELL 8: Set Neo4j credentials
import os

os.environ['NEO4J_URI'] = 'neo4j+s://xxxxxxxx.databases.neo4j.io'  # ← Your Aura URI
os.environ['NEO4J_USER'] = 'neo4j'                                  # ← Your username
os.environ['NEO4J_PASSWORD'] = 'your-password-here'                  # ← Your password
os.environ['QWEN_API_URL'] = ''

print(f"NEO4J_URI  = {os.environ['NEO4J_URI']}")
print(f"NEO4J_USER = {os.environ['NEO4J_USER']}")
print("✅ Credentials set")

## 🤖 PHASE 6 — Load Qwen 3.5 4B

In [ ]:
# CELL 9: Load Qwen 3.5 4B
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen3.5-4B"

print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
print(f"✅ {MODEL_NAME} loaded on {model.device}")

In [ ]:
# CELL 10: Generation function + test
import json

SYSTEM_PROMPT = """You are a precise technical assistant for semiconductor datasheets.
You receive structured data extracted from a knowledge graph and must format it into
a clear, accurate natural language answer.
RULES:
1. ONLY use the data provided — DO NOT add any information.
2. If data is missing, say "not found in the datasheet".
3. Always include units and conditions when available.
4. Never guess or approximate values.
"""

def generate_answer(query: str, data: dict, system_prompt: str = "") -> str:
    if not system_prompt:
        system_prompt = SYSTEM_PROMPT
    user_message = (
        f"User question: {query}\n\n"
        f"Knowledge Graph Data:\n{json.dumps(data, indent=2)}\n\n"
        f"Format a clear, precise answer using ONLY the data above."
    )
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_message},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=512, temperature=0.1, top_p=0.9, do_sample=True)
    generated = outputs[0][inputs.input_ids.shape[-1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

test = generate_answer("What is the supply voltage?", {"parameter": "VCC", "min": "2.7", "max": "5.5", "unit": "V"})
print(f"✅ Test: {test}")

## 🚀 PHASE 7 — Patch ai_client + Start Server with nohup

In [ ]:
# CELL 11: Patch ai_client for in-process Qwen
ai_client_code = '''
import logging
logger = logging.getLogger(__name__)
_generate_fn = None

def set_generate_function(fn):
    global _generate_fn
    _generate_fn = fn

async def format_with_qwen(query: str, structured_data: dict) -> str:
    if _generate_fn is None:
        logger.warning("Qwen not loaded")
        return ""
    try:
        return _generate_fn(query, structured_data)
    except Exception as e:
        logger.error("Qwen error: %s", e)
        return ""
'''

with open('backend/app/services/ai_client.py', 'w') as f:
    f.write(ai_client_code)
print("✅ ai_client.py patched")

In [ ]:
# CELL 12: Wire Qwen to backend
from backend.app.services.ai_client import set_generate_function
set_generate_function(generate_answer)
print("✅ Qwen wired to backend")

In [ ]:
# CELL 13: Start ngrok tunnel
NGROK_AUTH_TOKEN = "YOUR_NGROK_TOKEN_HERE"  # ← REPLACE!

from pyngrok import ngrok
ngrok.kill()
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
public_url = ngrok.connect(8000)

print("=" * 60)
print(f"🌐 PUBLIC URL: {public_url}")
print("=" * 60)
print(f"  {public_url}/health")
print(f"  {public_url}/api/upload")
print(f"  {public_url}/api/query")

In [ ]:
# CELL 14: Kill any old uvicorn process, then start fresh with nohup
import os

PROJECT_ROOT = '/content/drive/MyDrive/Knowledge_graph'

# Kill any existing uvicorn
!pkill -f uvicorn 2>/dev/null

# Start with nohup — PYTHONPATH is the critical fix
!nohup bash -c "cd {PROJECT_ROOT} && PYTHONPATH={PROJECT_ROOT}:$PYTHONPATH uvicorn backend.main:app --host 0.0.0.0 --port 8000" > {PROJECT_ROOT}/backend.log 2>&1 &

import time
time.sleep(3)

# Check if it started
!tail -n 20 {PROJECT_ROOT}/backend.log

In [ ]:
# CELL 15: Quick health check
import requests
try:
    r = requests.get('http://localhost:8000/health', timeout=5)
    print(f"✅ Server is UP: {r.json()}")
except Exception as e:
    print(f"❌ Server not responding: {e}")
    print("\nCheck logs:")
    !tail -n 30 /content/drive/MyDrive/Knowledge_graph/backend.log

## 🛠️ Utility — Restart / Check Logs

In [ ]:
# Run this cell anytime to check server logs
!tail -n 50 /content/drive/MyDrive/Knowledge_graph/backend.log

In [ ]:
# Run this cell to kill and restart the server
!pkill -f uvicorn 2>/dev/null
import time
time.sleep(1)
PROJECT_ROOT = '/content/drive/MyDrive/Knowledge_graph'
!nohup bash -c "cd {PROJECT_ROOT} && PYTHONPATH={PROJECT_ROOT}:$PYTHONPATH uvicorn backend.main:app --host 0.0.0.0 --port 8000" > {PROJECT_ROOT}/backend.log 2>&1 &
time.sleep(3)
!tail -n 10 {PROJECT_ROOT}/backend.log